# Impact du vieillissement démographique sur les dépenses de santé

Notebook principal — structure calquée sur le rapport :
1. Chargement & construction du panel
2. **Section 5.2** — OLS avec effets fixes année uniquement
3. **Section 5.3** — TWFE (effets fixes pays + année)

Le panel dynamique (Blundell-Bond) est estimé dans `main_R.r`.

## 0. Installation des dépendances

In [ ]:
!pip install openpyxl missingno statsmodels --quiet

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Chargement des données

Sources (dossier `data/`) :
- **Dépenses de santé** en % du PIB : OECD Health Statistics + WHO GHED
- **PIB par habitant**, part des +65 ans, population totale : Eurostat
- **Médecins** et lits pour 1000 hab. : OCDE
- **Taux de décès sur 2 ans** (proxy time-to-death) : construit dans `Mortalite_2ans.ipynb`
- **Taux de chômage** : Eurostat


In [ ]:
depenses_sante_PIB = pd.read_excel(os.path.join('data', 'Depenses_Sante_PIB_OCDE_fr.xlsx'))
pib_par_habitant   = pd.read_excel(os.path.join('data', 'PIB_par_habitant_fr.xlsx'))
part_pop65         = pd.read_excel(os.path.join('data', 'part_pop_plus_65_fr.xlsx'))
active_physicians  = pd.read_excel(os.path.join('data', 'active_physicians.xlsx'))
hospital_beds      = pd.read_excel(os.path.join('data', 'hospital_beds.xlsx'))
taux_deces_2ans    = pd.read_excel(os.path.join('data', 'Taux_Deces_2ans.xlsx'), index_col=None)
taux_deces_2ans    = taux_deces_2ans.rename(columns={taux_deces_2ans.columns[0]: 'TIME'})
chom               = pd.read_excel(os.path.join('data', 'taux_chomage_panel.xlsx'))


## 3. Construction du panel

Chaque fichier est au format *wide* (pays en lignes, années en colonnes). On le convertit en format *long* (Country × Year) puis on fusionne toutes les variables dans un panel unique.


In [ ]:
def preparer_donnees(fichier, var):
    """Convertit un fichier wide en format long indexé par (Country, Year)."""
    df = fichier.copy()
    df = df.replace(':', np.nan)
    df = pd.melt(df, id_vars=['TIME'], var_name='Year', value_name=var)
    df = df.rename(columns={'TIME': 'Country'})
    df['Year'] = df['Year'].astype(int)
    return df.set_index(['Country', 'Year']).sort_index()

# Fusion successive des variables
panel = preparer_donnees(depenses_sante_PIB, 'Depenses de sante en % du PIB')
panel = panel.join(preparer_donnees(pib_par_habitant,  'PIB par habitant'),          how='outer')
panel = panel.join(preparer_donnees(part_pop65,         'Part des +65 ans'),           how='outer')
panel = panel.join(preparer_donnees(active_physicians,  'Practiciens pour 1000 hab'),  how='outer')
panel = panel.join(preparer_donnees(hospital_beds,      'Lits pour 1000 hab'),         how='outer')
panel = panel.join(preparer_donnees(taux_deces_2ans,    'Taux deces 2 ans'),           how='outer')
panel = panel.join(preparer_donnees(chom,               'Chomage'),                    how='outer')

# Variable dépendante retardée d'un an (inertie budgétaire — cf. section 6 du rapport)
panel['Depenses t-1'] = panel.groupby(level='Country')['Depenses de sante en % du PIB'].shift(1)


## 4. Filtrage de l'échantillon

**Période retenue : 2005–2023** (20 années, 21 pays UE-OCDE).  
La Suède est exclue faute de données suffisantes sur toute la période.


In [ ]:
annees_a_supprimer = list(range(1993, 2005)) + [2024, 2025]
panel = panel.drop(annees_a_supprimer, level='Year', errors='ignore')
panel = panel.drop('Suède', level='Country', errors='ignore')

print(f"Pays  : {panel.index.get_level_values('Country').nunique()}")
print(f"Annees: {panel.index.get_level_values('Year').nunique()}")
print(f"Obs.  : {len(panel)}")


## 5. Gestion des valeurs manquantes

Imputation par la **médiane intra-pays** : on remplace les NaN de chaque pays par la médiane de ce même pays sur toute la période. Cela préserve l'hétérogénéité entre pays tout en évitant de perdre des observations.  

Le panel imputé est exporté en `panelV2.csv` pour être lu par `main_R.r`.


In [ ]:
cols_a_imputer = [
    'Depenses de sante en % du PIB', 'PIB par habitant', 'Part des +65 ans',
    'Practiciens pour 1000 hab', 'Lits pour 1000 hab', 'Taux deces 2 ans', 'Chomage'
]

panel_transformed = panel.copy()
for col in cols_a_imputer:
    panel_transformed[col] = (
        panel_transformed.groupby('Country')[col]
        .transform(lambda x: x.fillna(x.median()))
    )

# Export pour main_R.r (Blundell-Bond)
panel_transformed.to_csv('panelV2.csv', index=True)
print('panelV2.csv exporté.')


## 6. Fonctions d'estimation

On implémente deux estimateurs OLS avec **erreurs standard clustérisées par pays** (correction de l'autocorrélation intra-pays) :

- `ols_fe_year` : effets fixes année uniquement → section 5.2
- `twfe` : effets fixes pays **et** année → section 5.3

Les deux partagent la même fonction interne `_clustered_ols`.


In [ ]:
def _clustered_ols(Y, X, countries):
    """OLS avec erreurs clustérisées par pays (sandwich estimator)."""
    k     = X.shape[1]
    G     = len(np.unique(countries))
    N     = len(Y)
    beta  = np.linalg.lstsq(X, Y, rcond=None)[0]
    resid = Y - X @ beta
    # Matrice 'meat' du sandwich, agrégée par cluster (pays)
    meat = np.zeros((k, k))
    for c in np.unique(countries):
        idx = countries == c
        sc  = X[idx].T @ resid[idx]
        meat += np.outer(sc, sc)
    corr    = G / (G - 1) * (N - 1) / (N - k)
    XtX_inv = np.linalg.inv(X.T @ X)
    se      = np.sqrt(np.diag(corr * XtX_inv @ meat @ XtX_inv))
    t       = beta / se
    p       = 2 * (1 - stats.t.cdf(np.abs(t), df=G - 1))
    r2      = 1 - np.sum(resid**2) / np.sum((Y - Y.mean())**2)
    return beta, se, p, r2, N


def ols_fe_year(data, regressors, dep):
    """Section 5.2 : FE année uniquement.
    Déméaning sur l'année -> ne neutralise pas l'hétérogénéité entre pays."""
    cols = ['Country', 'Year', dep] + regressors
    sub  = data[cols].dropna().copy().sort_values(['Country', 'Year'])
    for col in [dep] + regressors:
        sub[col + '_fe'] = (
            sub[col]
            - sub.groupby('Year')[col].transform('mean')
            + sub[col].mean()
        )
    Y = sub[dep + '_fe'].values
    X = sub[[r + '_fe' for r in regressors]].values
    return _clustered_ols(Y, X, sub['Country'].values)


def twfe(data, regressors, dep):
    """Section 5.3 : TWFE — FE pays + année.
    Déméaning sur pays ET année -> n'exploite que la variation within-pays."""
    cols = ['Country', 'Year', dep] + regressors
    sub  = data[cols].dropna().copy().sort_values(['Country', 'Year'])
    for col in [dep] + regressors:
        sub[col + '_fe'] = (
            sub[col]
            - sub.groupby('Year')[col].transform('mean')
            - sub.groupby('Country')[col].transform('mean')
            + sub[col].mean()
        )
    Y = sub[dep + '_fe'].values
    X = sub[[r + '_fe' for r in regressors]].values
    return _clustered_ols(Y, X, sub['Country'].values)


def afficher_tableau(titre, fe_pays, res1, res2, M1_REGS, M2_REGS, all_vars, labels):
    """Affiche un tableau de résultats sur deux modèles."""
    b1, se1, p1, r2_1, n1 = res1
    b2, se2, p2, r2_2, n2 = res2

    def sig(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        return 'ns'

    W = 70
    print(f'\n{"="*W}')
    print(f'  {titre}')
    print(f'{"="*W}')
    print(f'  {"Variable":<28} {"Modele 1":>18} {"Modele 2":>18}')
    print(f'  {"-"*68}')
    for var, lab in zip(all_vars, labels):
        if var in M1_REGS:
            i1  = M1_REGS.index(var)
            c1  = f'{b1[i1]:.4f} {sig(p1[i1])}'
            s1  = f'({se1[i1]:.4f})'
            pv1 = f'[p={p1[i1]:.3f}]'
        else:
            c1, s1, pv1 = '--', '', ''
        i2  = M2_REGS.index(var)
        c2  = f'{b2[i2]:.4f} {sig(p2[i2])}'
        s2  = f'({se2[i2]:.4f})'
        pv2 = f'[p={p2[i2]:.3f}]'
        print(f'  {lab:<28} {c1:>18} {c2:>18}')
        print(f'  {"":28} {s1:>18} {s2:>18}')
        print(f'  {"":28} {pv1:>18} {pv2:>18}')
        print()
    print(f'  {"-"*68}')
    print(f'  {"FE pays":<28} {fe_pays:>18} {fe_pays:>18}')
    print(f'  {"FE annee":<28} {"Oui":>18} {"Oui":>18}')
    print(f'  {"Observations":<28} {n1:>18} {n2:>18}')
    print(f'  {"R2 within":<28} {r2_1:>18.4f} {r2_2:>18.4f}')
    print(f'{"="*W}')
    print('  SE clusterisees par pays.  * p<0.05  ** p<0.01  *** p<0.001')


## 7. Section 5.2 — OLS avec effets fixes année uniquement (Table 1)

**Modèle 1** : Part65 + PIB/hab + Chômage  
**Modèle 2** : Part65 + TxDeces + PIB/hab + Chômage  

> ⚠️ Sans effets fixes pays, la relation positive observée reflète surtout le fait que les pays historiquement plus vieux ont des systèmes de santé plus développés. Ce n'est pas un effet causal — c'est corrigé en section 5.3.


In [ ]:
DEP     = 'Depenses de sante en % du PIB'
M1_REGS = ['Part des +65 ans', 'PIB par habitant', 'Chomage']
M2_REGS = ['Part des +65 ans', 'Taux deces 2 ans', 'PIB par habitant', 'Chomage']

all_vars = ['Part des +65 ans', 'Taux deces 2 ans', 'PIB par habitant', 'Chomage']
labels   = ['Part des +65 ans', 'Taux de deces <2 ans', 'PIB par habitant', 'Taux de chomage']

# On remet le MultiIndex à plat pour les fonctions d'estimation
panel_flat = panel.reset_index()

res1_fe = ols_fe_year(panel_flat, M1_REGS, DEP)
res2_fe = ols_fe_year(panel_flat, M2_REGS, DEP)

afficher_tableau(
    titre    = 'OLS -- effets fixes annee uniquement (Table 1 du rapport)',
    fe_pays  = 'Non',
    res1     = res1_fe,
    res2     = res2_fe,
    M1_REGS  = M1_REGS,
    M2_REGS  = M2_REGS,
    all_vars = all_vars,
    labels   = labels
)


## 8. Section 5.3 — TWFE : effets fixes pays + année (Table 2)

On ajoute les **effets fixes pays** pour neutraliser toute l'hétérogénéité structurelle stable dans le temps (système de santé, niveau de vie, institutions...).  
On n'exploite plus que **l'évolution de chaque pays au fil du temps**.

> Résultat attendu (cf. rapport) : le coefficient de Part65 perd sa significativité (passe de +41,7 à -2,6), confirmant que l'effet de section 5.2 était un artefact de coupe transversale et non un effet causal du vieillissement.


In [ ]:
res1_twfe = twfe(panel_flat, M1_REGS, DEP)
res2_twfe = twfe(panel_flat, M2_REGS, DEP)

afficher_tableau(
    titre    = 'TWFE -- effets fixes pays + annee (Table 2 du rapport)',
    fe_pays  = 'Oui',
    res1     = res1_twfe,
    res2     = res2_twfe,
    M1_REGS  = M1_REGS,
    M2_REGS  = M2_REGS,
    all_vars = all_vars,
    labels   = labels
)
